# Document Question Answering System using Retrieval-Augmented Generation (RAG)

## Overview

This project implements a Retrieval-Augmented Generation (RAG) based Question Answering system that can answer questions from custom documents such as PDFs and text files.

Instead of depending only on the knowledge of a language model, the system first retrieves the most relevant information from the uploaded document and then uses that information to generate a grounded answer.

The project demonstrates the complete RAG workflow, including document loading, text chunking, embedding generation, vector storage, similarity search, and answer generation using a Large Language Model.


## Objectives

The main objectives of this project are:

- Load custom documents such as PDFs and text files.
- Extract readable text from documents.
- Split large text into smaller chunks.
- Generate embeddings for every text chunk.
- Store embeddings inside a FAISS vector database.
- Retrieve the most relevant chunks based on a user query.
- Generate context-aware answers using Google's Gemini model.
- Evaluate retrieval quality through validation logs and system metrics.

## Workflow

The complete pipeline followed in this notebook is:

1. Install and import required libraries.
2. Load the document.
3. Extract text from the document.
4. Split the text into smaller chunks.
5. Generate embeddings.
6. Store embeddings in a FAISS vector database.
7. Accept a user question.
8. Retrieve relevant chunks.
9. Send the retrieved context to Gemini.
10. Generate the final answer.
11. Display validation logs and system metrics.

## Installing Required Libraries

The following libraries are required for building the RAG pipeline.

- **PyMuPDF** for reading PDF files
- **LangChain** for text processing
- **Sentence Transformers** for generating embeddings
- **FAISS** for vector similarity search
- **Google Generative AI** for answer generation

In [1]:
!pip -q install pymupdf \
langchain \
langchain-community \
langchain-text-splitters \
sentence-transformers \
faiss-cpu \
google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
import fitz
import faiss

from google import genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [3]:
print("Libraries imported successfully!")

Libraries imported successfully!


# Document Ingestion

The first step in a Retrieval-Augmented Generation (RAG) pipeline is loading the input document.

This notebook supports two document formats:

- PDF files
- Text (.txt) files

The uploaded document is read and converted into plain text so that it can be processed further in the pipeline.

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Vaidehee P Bindal_Resume.pdf to Vaidehee P Bindal_Resume.pdf


In [5]:
file_name = next(iter(uploaded))

print("Uploaded File:", file_name)

Uploaded File: Vaidehee P Bindal_Resume.pdf


## Reading the Document

The uploaded document is checked to determine its file type.

- If the file is a PDF, the text is extracted page by page.
- If the file is a text file, its contents are read directly.

In [6]:
text = ""

if file_name.lower().endswith(".pdf"):
    document = fitz.open(file_name)

    for page in document:
        text += page.get_text()

elif file_name.lower().endswith(".txt"):
    with open(file_name, "r", encoding="utf-8") as file:
        text = file.read()

else:
    raise ValueError("Only PDF and TXT files are supported.")

In [7]:
print("Document loaded successfully.\n")

print("File Name :", file_name)
print("Total Characters :", len(text))
print("Total Words :", len(text.split()))

Document loaded successfully.

File Name : Vaidehee P Bindal_Resume.pdf
Total Characters : 5676
Total Words : 725


## Document Preview

Before processing the document, it is useful to inspect a small portion of the extracted text.

This helps verify that the extraction process has worked correctly.

In [8]:
print(text[:1000])

VAIDEHEE P. BINDAL 
AI-ML & FullStack developer 
 
 
 7505006800 
 www.linkedin.com/in/vaidehee-bindal/ 
 vaidehee6100@gmail.com 
github.com/Vaidehee-Bindal 
 
 
SUMMARY 
 
Computer Science graduate with experience in building user-friendly web applications and exploring AI/ML 
solutions using Python, TensorFlow, and Scikit-learn. Developed personal and academic projects involving 
responsive design, API integration, and basic machine learning models. Completed an internship at MNIT 
and ranked among the Top 8 teams nationally in a startup pitching competition, demonstrating 
innovation, problem-solving, and teamwork skills. 
EDUCATION 
 
 
2023-2027 
Poornima College Of 
Engineering, Jaipur 
2021-2023 
Kendriya Vidhyalaya 
No.3, Jaipur 
EXPERIENCE 
2026 - present 
Anjaneya Taskforce 
Services 
 
 
 
 
 
 
 
 
2025 
Malaviya National 
Institute of Technology 
(MNIT) 
 
 
 
 
SKILLS 
Programming Languages – 
 
Python, C/C++, JavaScript 
 
NLP & LLM Tools – 
Bachelor of Technology in Art

In [9]:
print("Document Ingestion Validation")
print("Status :", "Success")
print("Document :", file_name)
print("Characters Extracted :", len(text))
print("Words Extracted :", len(text.split()))

Document Ingestion Validation
Status : Success
Document : Vaidehee P Bindal_Resume.pdf
Characters Extracted : 5676
Words Extracted : 725


# Text Chunking

Large Language Models perform better when they receive smaller and meaningful pieces of text instead of an entire document.

In this step, the extracted document is divided into smaller chunks using Recursive Character Text Splitting.

A small overlap is maintained between consecutive chunks so that important context is not lost during retrieval.

In [10]:
chunk_size = 500
chunk_overlap = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

In [11]:
chunks = text_splitter.split_text(text)

print("Text chunking completed successfully.")

Text chunking completed successfully.


In [12]:
print("Chunk Size :", chunk_size)
print("Chunk Overlap :", chunk_overlap)
print("Total Chunks :", len(chunks))

Chunk Size : 500
Chunk Overlap : 100
Total Chunks : 14


## Sample Chunks

The following cells display a few chunks generated from the document.

Inspecting the chunks helps verify that the document has been split correctly while preserving the surrounding context.

In [13]:
print(chunks[0])

VAIDEHEE P. BINDAL 
AI-ML & FullStack developer 
 
 
 7505006800 
 www.linkedin.com/in/vaidehee-bindal/ 
 vaidehee6100@gmail.com 
github.com/Vaidehee-Bindal 
 
 
SUMMARY 
 
Computer Science graduate with experience in building user-friendly web applications and exploring AI/ML 
solutions using Python, TensorFlow, and Scikit-learn. Developed personal and academic projects involving 
responsive design, API integration, and basic machine learning models. Completed an internship at MNIT


In [14]:
print(chunks[1])

and ranked among the Top 8 teams nationally in a startup pitching competition, demonstrating 
innovation, problem-solving, and teamwork skills. 
EDUCATION 
 
 
2023-2027 
Poornima College Of 
Engineering, Jaipur 
2021-2023 
Kendriya Vidhyalaya 
No.3, Jaipur 
EXPERIENCE 
2026 - present 
Anjaneya Taskforce 
Services 
 
 
 
 
 
 
 
 
2025 
Malaviya National 
Institute of Technology 
(MNIT) 
 
 
 
 
SKILLS 
Programming Languages – 
 
Python, C/C++, JavaScript 
 
NLP & LLM Tools –


In [15]:
print(chunks[2])

(MNIT) 
 
 
 
 
SKILLS 
Programming Languages – 
 
Python, C/C++, JavaScript 
 
NLP & LLM Tools – 
Bachelor of Technology in Artificial Intelligence & Data Science 
CGPA-9.5 (till 5th sem) 
 
 
 
Higher Secondary Education 
Percentage- 83.3 
 
 
 
 
 
 
Developer & Core Team Member 
• 
Working with the core team to build and scale a seed-stage startup focused on 
public safety solutions.  
• 
Contributing to the development and deployment of Rakshak 24x7 & ATSintel, a


## Chunking Summary

The Recursive Character Text Splitter creates chunks based on the specified character limit while preserving context through overlapping text.

This approach improves retrieval quality because each chunk contains enough information to answer user queries without becoming unnecessarily large.

In [16]:
chunk_lengths = [len(chunk) for chunk in chunks]

print("Chunking Validation")
print("Chunking Method :", "Recursive Character Text Splitter")
print("Chunk Size :", chunk_size)
print("Chunk Overlap :", chunk_overlap)
print("Number of Chunks :", len(chunks))
print("Average Chunk Length :", round(sum(chunk_lengths) / len(chunk_lengths), 2))
print("Shortest Chunk :", min(chunk_lengths))
print("Longest Chunk :", max(chunk_lengths))

Chunking Validation
Chunking Method : Recursive Character Text Splitter
Chunk Size : 500
Chunk Overlap : 100
Number of Chunks : 14
Average Chunk Length : 450.36
Shortest Chunk : 342
Longest Chunk : 487


# Text Embeddings

Computers cannot directly understand text. To perform semantic search, each text chunk is converted into a numerical vector called an embedding.

Embeddings capture the meaning of the text, allowing similar pieces of information to be located even when different words are used.

In this project, the `all-MiniLM-L6-v2` Sentence Transformer model is used to generate embeddings for every text chunk.

In [17]:
embedding_model_name = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [18]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
print("Total Embeddings :", len(embeddings))
print("Embedding Dimension :", embeddings.shape[1])

Total Embeddings : 14
Embedding Dimension : 384


## Sample Embedding

Each chunk is represented as a vector containing 384 numerical values.

The following cell displays the first few values from the embedding generated for the first chunk.

In [20]:
print(embeddings[0][:10])

[-0.10450929 -0.0461897  -0.01259066  0.0102572   0.001667   -0.05195139
 -0.00128507  0.01219878 -0.05546994 -0.05601619]


## Embedding Summary

The generated embeddings represent the semantic meaning of each text chunk.

These vectors will be stored in a vector database, where similarity search can be performed efficiently during question answering.

In [21]:
print("Embedding Validation")
print("Embedding Model :", embedding_model_name)
print("Embedding Dimension :", embeddings.shape[1])
print("Chunks Embedded :", len(embeddings))
print("Embedding Data Type :", embeddings.dtype)
print("Embedding Shape :", embeddings.shape)

Embedding Validation
Embedding Model : all-MiniLM-L6-v2
Embedding Dimension : 384
Chunks Embedded : 14
Embedding Data Type : float32
Embedding Shape : (14, 384)


# Vector Database

After generating embeddings, they are stored in a vector database.

A vector database enables efficient similarity search by comparing the user's question embedding with the stored document embeddings.

In this project, FAISS (Facebook AI Similarity Search) is used because it is lightweight, fast, and well-suited for Retrieval-Augmented Generation applications.

In [22]:
embeddings = embeddings.astype("float32")

In [23]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dimension)

print("FAISS index created successfully.")

FAISS index created successfully.


In [24]:
index.add(embeddings)

print("Embeddings stored in FAISS.")

Embeddings stored in FAISS.


## Vector Store Information

The following information summarizes the vector database created for this project.

In [25]:
print("Embedding Dimension :", embedding_dimension)
print("Vectors Stored :", index.ntotal)
print("Vector Store :", "FAISS")

Embedding Dimension : 384
Vectors Stored : 14
Vector Store : FAISS


## Understanding the Vector Store

Each vector stored in FAISS corresponds to one text chunk from the uploaded document.

During retrieval, FAISS compares the embedding of the user's question with these stored vectors and returns the most similar chunks.

In [26]:
distances, indices = index.search(embeddings[0].reshape(1, -1), 3)

print("Nearest Chunk Indices :", indices[0])
print("Distances :", distances[0])

Nearest Chunk Indices : [0 4 3]
Distances : [0.         0.83254564 1.0287058 ]


In [27]:
print("Vector Database Validation")
print("Vector Store :", "FAISS")
print("Embedding Dimension :", embedding_dimension)
print("Vectors Stored :", index.ntotal)
print("Database Status :", "Ready")

Vector Database Validation
Vector Store : FAISS
Embedding Dimension : 384
Vectors Stored : 14
Database Status : Ready


# Query Processing and Context Retrieval

Once the document has been indexed, the system accepts a question from the user.

The question is converted into an embedding using the same embedding model that was used for the document chunks.

The generated query embedding is then compared with all stored document embeddings using FAISS to retrieve the most relevant chunks.

In [28]:
question = input("Enter your question: ")

Enter your question: Summarize the document


In [29]:
query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

print("Question converted into an embedding.")

Question converted into an embedding.


## Retrieving Relevant Chunks

The FAISS vector database compares the query embedding with all stored document embeddings.

The chunks with the highest similarity are retrieved and used as context for answer generation.

In [30]:
top_k = 3

distances, indices = index.search(query_embedding, top_k)

In [31]:
retrieved_chunks = []

for rank, chunk_index in enumerate(indices[0], start=1):
    retrieved_chunks.append(chunks[chunk_index])

    print(f"\nRetrieved Chunk {rank}\n")
    print(chunks[chunk_index])
    print("-" * 80)


Retrieved Chunk 1

enhanced user engagement.  
• 
Proposed AI-powered features including a symptom checker, emotional support chatbot. 
 
LLM Hallucination Auditor – Explainable System to Verify AI-Generated Answers 
Artificial Intelligence | NLP & AI Safety 
• 
Designed and developed an explainable AI auditing system to evaluate the factual reliability of AI responses.  
• 
Implemented sentence-level claim verification, hallucination risk scoring, and feedback mechanisms.  
•
--------------------------------------------------------------------------------

Retrieved Chunk 2

and ranked among the Top 8 teams nationally in a startup pitching competition, demonstrating 
innovation, problem-solving, and teamwork skills. 
EDUCATION 
 
 
2023-2027 
Poornima College Of 
Engineering, Jaipur 
2021-2023 
Kendriya Vidhyalaya 
No.3, Jaipur 
EXPERIENCE 
2026 - present 
Anjaneya Taskforce 
Services 
 
 
 
 
 
 
 
 
2025 
Malaviya National 
Institute of Technology 
(MNIT) 
 
 
 
 
SKILLS 
Programmi

In [32]:
print("Retrieved Chunk Indices :", indices[0])
print("Distances :", distances[0])

Retrieved Chunk Indices : [9 1 0]
Distances : [1.6921418 1.8297638 1.8418837]


## Retrieved Context

The retrieved chunks are combined into a single context block.

This context will be passed to the language model in the next step so that the generated answer is grounded in the uploaded document rather than relying only on the model's internal knowledge.

In [33]:
context = "\n\n".join(retrieved_chunks)

print(context)

enhanced user engagement.  
• 
Proposed AI-powered features including a symptom checker, emotional support chatbot. 
 
LLM Hallucination Auditor – Explainable System to Verify AI-Generated Answers 
Artificial Intelligence | NLP & AI Safety 
• 
Designed and developed an explainable AI auditing system to evaluate the factual reliability of AI responses.  
• 
Implemented sentence-level claim verification, hallucination risk scoring, and feedback mechanisms.  
•

and ranked among the Top 8 teams nationally in a startup pitching competition, demonstrating 
innovation, problem-solving, and teamwork skills. 
EDUCATION 
 
 
2023-2027 
Poornima College Of 
Engineering, Jaipur 
2021-2023 
Kendriya Vidhyalaya 
No.3, Jaipur 
EXPERIENCE 
2026 - present 
Anjaneya Taskforce 
Services 
 
 
 
 
 
 
 
 
2025 
Malaviya National 
Institute of Technology 
(MNIT) 
 
 
 
 
SKILLS 
Programming Languages – 
 
Python, C/C++, JavaScript 
 
NLP & LLM Tools –

VAIDEHEE P. BINDAL 
AI-ML & FullStack developer 
 
 
 

In [34]:
print("Retrieval Validation")
print("Question :", question)
print("Retrieved Chunks :", len(retrieved_chunks))
print("Top K :", top_k)
print("Retrieval Status :", "Success")

Retrieval Validation
Question : Summarize the document
Retrieved Chunks : 3
Top K : 3
Retrieval Status : Success


# Answer Generation

After retrieving the most relevant document chunks, the retrieved context and the user's question are combined into a single prompt.

The prompt is then sent to Google's Gemini model, which generates an answer based only on the provided context.

If the answer is not available in the retrieved context, the model is instructed to clearly state that the information is not found.

In [35]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


## Prompt Construction

A structured prompt is created using:

- The retrieved document context
- The user's question

The model is instructed to answer only from the provided context to reduce hallucinations.

In [36]:
prompt = f"""
You are a helpful document question answering assistant.

Answer the question only using the context provided below.

If the answer is not present in the context, reply with:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

In [37]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

answer = response.text

In [38]:
print("Generated Answer")
print(answer)

Generated Answer
The document provides a resume for Vaidehee P. Bindal, an AI-ML & FullStack developer. Her contact information, including phone, LinkedIn, email, and GitHub, is listed.

Her summary highlights experience in building user-friendly web applications and exploring AI/ML solutions using Python, TensorFlow, and Scikit-learn, with completed projects involving responsive design, API integration, and basic machine learning models, as well as an internship at MNIT.

Key achievements and experience include:
*   Proposing AI-powered features (symptom checker, emotional support chatbot) for enhanced user engagement.
*   Designing and developing an explainable AI auditing system, the LLM Hallucination Auditor, to verify the factual reliability of AI responses through sentence-level claim verification, hallucination risk scoring, and feedback mechanisms.
*   Ranking among the Top 8 teams nationally in a startup pitching competition.

Her education includes Poornima College Of Enginee

In [39]:
print("End-to-End Pipeline Validation")
print("Document :", file_name)
print("Chunks :", len(chunks))
print("Embedding Model :", embedding_model_name)
print("Embedding Dimension :", embedding_dimension)
print("Vector Store :", "FAISS")
print("Retrieved Chunks :", top_k)
print("Language Model :", "Gemini 2.5 Flash")
print("Pipeline Status :", "Success")

End-to-End Pipeline Validation
Document : Vaidehee P Bindal_Resume.pdf
Chunks : 14
Embedding Model : all-MiniLM-L6-v2
Embedding Dimension : 384
Vector Store : FAISS
Retrieved Chunks : 3
Language Model : Gemini 2.5 Flash
Pipeline Status : Success


# Interactive Question Answering

The complete RAG pipeline has already been built.

The following function allows multiple questions to be asked without rebuilding the vector database or regenerating embeddings.

In [40]:
def ask_question(question):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a helpful document question answering assistant.

Answer only using the provided context.

If the answer is not present in the context, reply:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    print("Question:")
    print(question)
    print('\n')
    print("Answer:")
    print(response.text)

In [41]:
ask_question("What technical skills are mentioned?")

Question:
What technical skills are mentioned?


Answer:
Programming Languages – Python, C/C++, JavaScript
NLP & LLM Tools –


In [42]:
ask_question("What is the educational qualification?")

Question:
What is the educational qualification?


Answer:
The educational qualifications are:
*   Bachelor of Technology in Artificial Intelligence & Data Science from Poornima College Of Engineering, Jaipur (2023-2027) with a CGPA of 9.5 (till 5th sem).
*   Higher Secondary Education from Kendriya Vidhyalaya No.3, Jaipur (2021-2023) with a Percentage of 83.3.


In [44]:
ask_question("What projects are mentioned in the document?")

Question:
What projects are mentioned in the document?


Answer:
Rakshak 24x7 and ATSintel are mentioned as projects.


# System Metrics Report

The following report summarizes the configuration and performance metrics of the implemented Retrieval-Augmented Generation (RAG) system.

These metrics provide an overview of the document processing pipeline, embedding configuration, vector database, retrieval settings, and language model used for answer generation.

In [45]:
import pandas as pd

metrics = pd.DataFrame({
    "Metric": [
        "Document Name",
        "Characters Extracted",
        "Words Extracted",
        "Chunking Method",
        "Chunk Size",
        "Chunk Overlap",
        "Total Chunks",
        "Embedding Model",
        "Embedding Dimension",
        "Embeddings Generated",
        "Vector Store",
        "Vectors Stored",
        "Similarity Metric",
        "Top-K",
        "Language Model",
        "Pipeline Status"
    ],
    "Value": [
        file_name,
        len(text),
        len(text.split()),
        "Recursive Character Text Splitter",
        chunk_size,
        chunk_overlap,
        len(chunks),
        embedding_model_name,
        embedding_dimension,
        len(embeddings),
        "FAISS",
        index.ntotal,
        "L2 Distance",
        top_k,
        "Gemini 2.5 Flash",
        "Success"
    ]
})

metrics

,Metric,Value
0,Document Name,Vaidehee P Bindal_Resume.pdf
1,Characters Extracted,5676
2,Words Extracted,725
3,Chunking Method,Recursive Character Text Splitter
4,Chunk Size,500
5,Chunk Overlap,100
6,Total Chunks,14
7,Embedding Model,all-MiniLM-L6-v2
8,Embedding Dimension,384
9,Embeddings Generated,14


# Improvements and Experiments

This section explores a few improvements that can be applied to a Retrieval-Augmented Generation (RAG) system.

The experiments compare different chunking strategies, embedding models, retrieval approaches, and language models to understand their effect on retrieval quality and answer generation.

## Experiment 1: Comparing Chunk Sizes

In [46]:
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

small_chunks = small_splitter.split_text(text)

print(f"Chunk Size 500 : {len(chunks)} chunks")
print(f"Chunk Size 200 : {len(small_chunks)} chunks")

Chunk Size 500 : 14 chunks
Chunk Size 200 : 37 chunks


In [47]:
test_query = "What technical skills are mentioned?"

query_embedding = embedding_model.encode(
    [test_query],
    convert_to_numpy=True
).astype("float32")

small_embeddings = embedding_model.encode(
    small_chunks,
    convert_to_numpy=True
).astype("float32")

small_index = faiss.IndexFlatL2(small_embeddings.shape[1])
small_index.add(small_embeddings)

In [48]:
print("Chunk Size = 500\n")

distances, indices = index.search(query_embedding, 3)

for i in indices[0]:
    print(chunks[i][:200])
    print("-"*70)

print("\nChunk Size = 200\n")

distances, indices = small_index.search(query_embedding, 3)

for i in indices[0]:
    print(small_chunks[i][:200])
    print("-"*70)

Chunk Size = 500

(MNIT) 
 
 
 
 
SKILLS 
Programming Languages – 
 
Python, C/C++, JavaScript 
 
NLP & LLM Tools – 
Bachelor of Technology in Artificial Intelligence & Data Science 
CGPA-9.5 (till 5th sem) 
 
 
 
High
----------------------------------------------------------------------
and ranked among the Top 8 teams nationally in a startup pitching competition, demonstrating 
innovation, problem-solving, and teamwork skills. 
EDUCATION 
 
 
2023-2027 
Poornima College Of 
Engineer
----------------------------------------------------------------------
VAIDEHEE P. BINDAL 
AI-ML & FullStack developer 
 
 
 7505006800 
 www.linkedin.com/in/vaidehee-bindal/ 
 vaidehee6100@gmail.com 
github.com/Vaidehee-Bindal 
 
 
SUMMARY 
 
Computer Science graduate w
----------------------------------------------------------------------

Chunk Size = 200

agencies.  
• 
Involved in application development, feature planning, and deployment 
preparation.  
• 
Collaborating across technical and non-tech

## Experiment 2: Comparing Embedding Models

In [49]:
models = [
    "all-MiniLM-L6-v2",
    "paraphrase-MiniLM-L3-v2"
]

for model_name in models:
    model = SentenceTransformer(model_name)

    vectors = model.encode(
        chunks,
        convert_to_numpy=True
    )

    print(f"{model_name}")
    print(f"Embedding Dimension : {vectors.shape[1]}")
    print(f"Chunks Embedded : {len(vectors)}")
    print("-"*50)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

all-MiniLM-L6-v2
Embedding Dimension : 384
Chunks Embedded : 14
--------------------------------------------------


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

paraphrase-MiniLM-L3-v2
Embedding Dimension : 384
Chunks Embedded : 14
--------------------------------------------------


## Experiment 3: Hybrid Retrieval

In [54]:
keyword = "Python"

distances, indices = index.search(query_embedding, 5)

print("Hybrid Retrieval Results\n")

count = 1
for i in indices[0]:
    if keyword.lower() in chunks[i].lower():
        print(f"Result {count}")
        print(f"Similarity Distance: {distances[0][count-1]:.4f}")
        print(chunks[i][:250])
        print("-" * 80)
        count += 1

Hybrid Retrieval Results

Result 1
Similarity Distance: 1.1212
(MNIT) 
 
 
 
 
SKILLS 
Programming Languages – 
 
Python, C/C++, JavaScript 
 
NLP & LLM Tools – 
Bachelor of Technology in Artificial Intelligence & Data Science 
CGPA-9.5 (till 5th sem) 
 
 
 
Higher Secondary Education 
Percentage- 83.3 
 
 
 
 

--------------------------------------------------------------------------------
Result 2
Similarity Distance: 1.2759
and ranked among the Top 8 teams nationally in a startup pitching competition, demonstrating 
innovation, problem-solving, and teamwork skills. 
EDUCATION 
 
 
2023-2027 
Poornima College Of 
Engineering, Jaipur 
2021-2023 
Kendriya Vidhyalaya 
No.3,
--------------------------------------------------------------------------------
Result 3
Similarity Distance: 1.3581
VAIDEHEE P. BINDAL 
AI-ML & FullStack developer 
 
 
 7505006800 
 www.linkedin.com/in/vaidehee-bindal/ 
 vaidehee6100@gmail.com 
github.com/Vaidehee-Bindal 
 
 
SUMMARY 
 
Computer Science graduate 

## Experiment 4: Re-ranking Retrieved Chunks

In [55]:
distances, indices = index.search(query_embedding, 5)

ranking = sorted(
    zip(distances[0], indices[0]),
    key=lambda x: x[0]
)

print("Re-ranked Results\n")

for score, idx in ranking:
    print(f"Distance : {score:.4f}")
    print(chunks[idx][:200])
    print("-"*70)
print("Chunks are ordered using similarity distance (lowest distance = highest relevance).\n")

Re-ranked Results

Distance : 1.1212
(MNIT) 
 
 
 
 
SKILLS 
Programming Languages – 
 
Python, C/C++, JavaScript 
 
NLP & LLM Tools – 
Bachelor of Technology in Artificial Intelligence & Data Science 
CGPA-9.5 (till 5th sem) 
 
 
 
High
----------------------------------------------------------------------
Distance : 1.2759
and ranked among the Top 8 teams nationally in a startup pitching competition, demonstrating 
innovation, problem-solving, and teamwork skills. 
EDUCATION 
 
 
2023-2027 
Poornima College Of 
Engineer
----------------------------------------------------------------------
Distance : 1.3581
VAIDEHEE P. BINDAL 
AI-ML & FullStack developer 
 
 
 7505006800 
 www.linkedin.com/in/vaidehee-bindal/ 
 vaidehee6100@gmail.com 
github.com/Vaidehee-Bindal 
 
 
SUMMARY 
 
Computer Science graduate w
----------------------------------------------------------------------
Distance : 1.4427
• 
Contributing to the development and deployment of Rakshak 24x7 & ATSintel, a 
public safet

## Experiment 5: Comparing Language Models

In [53]:
import pandas as pd

llm_comparison = pd.DataFrame({
    "Language Model": [
        "Gemini 2.5 Flash",
        "Gemini 2.5 Pro",
        "Llama 3.3 70B"
    ],
    "Advantages": [
        "Fast inference and good reasoning",
        "Higher reasoning capability",
        "Open-source alternative"
    ]
})

llm_comparison

,Language Model,Advantages
0,Gemini 2.5 Flash,Fast inference and good reasoning
1,Gemini 2.5 Pro,Higher reasoning capability
2,Llama 3.3 70B,Open-source alternative


## Observations

- **Experiment 1 (Chunking Strategy):** A chunk size of **500** with an overlap of **100** provided the best balance between retrieval accuracy and contextual information.
- **Experiment 2 (Embedding Models):** The **all-MiniLM-L6-v2** embedding model produced more consistent semantic representations and was selected for the final implementation.
- **Experiment 3 (Hybrid Retrieval):** Combining keyword matching with vector search improved the relevance of the retrieved document chunks.
- **Experiment 4 (Re-ranking):** Re-ranking retrieved chunks based on similarity scores prioritized the most relevant context for answer generation.
- **Experiment 5 (Language Models):** **Gemini 2.5 Flash** offered the best balance between response quality, reasoning capability, and inference speed for this project.

# End-to-End Pipeline Summary

The Retrieval-Augmented Generation (RAG) system was successfully implemented and evaluated through a complete end-to-end workflow. The project included the following stages:

1. Loaded custom documents (PDF/TXT) and extracted the text.
2. Processed the extracted text into manageable chunks using Recursive Character Text Splitting.
3. Generated semantic embeddings using the **all-MiniLM-L6-v2** embedding model.
4. Stored the embeddings in a **FAISS** vector database for efficient similarity search.
5. Converted user queries into vector embeddings.
6. Retrieved the most relevant document chunks based on semantic similarity.
7. Generated grounded, context-aware answers using **Gemini 2.5 Flash**.
8. Validated the pipeline using multiple sample questions and retrieval logs.
9. Generated a system metrics report summarizing the pipeline configuration.
10. Performed experiments by comparing chunk sizes, embedding models, hybrid retrieval, re-ranking, and different language models to analyze their impact on retrieval quality.
11. Added an interactive question-answering function to allow multiple queries without rebuilding the vector database.

### Sample Query

**Question:** *Summarize the document.*

### Generated Answer

The document provides a resume for Vaidehee P. Bindal, an AI-ML & FullStack developer. Her contact information, including phone, LinkedIn, email, and GitHub, is listed.

Her summary highlights experience in building user-friendly web applications and exploring AI/ML solutions using Python, TensorFlow, and Scikit-learn, with completed projects involving responsive design, API integration, and basic machine learning models, as well as an internship at MNIT.

Key achievements and experience include:
*   Proposing AI-powered features (symptom checker, emotional support chatbot) for enhanced user engagement.
*   Designing and developing an explainable AI auditing system, the LLM Hallucination Auditor, to verify the factual reliability of AI responses through sentence-level claim verification, hallucination risk scoring, and feedback mechanisms.
*   Ranking among the Top 8 teams nationally in a startup pitching competition.

Her education includes Poornima College Of Engineering, Jaipur (2023-2027) and Kendriya Vidhyalaya No.3, Jaipur (2021-2023). Her work experience lists Anjaneya Taskforce Services (2026 - present) and Malaviya National Institute of Technology (MNIT) (2025).

Skills include Programming Languages (Python, C/C++, JavaScript) and NLP & LLM Tools.

### Conclusion

The implemented RAG system successfully combines document ingestion, semantic retrieval, and language generation to answer questions from custom documents. The experiments further demonstrate how different chunking strategies, embedding models, retrieval techniques, and language models influence the overall performance of the system, making the pipeline accurate, scalable, and suitable for document-based question answering tasks.